# Highlighted organ tree visualizations

This notebook is a non-destructive derivative of `HRA_OrganTree_v1.ipynb`.
It retains the original HRA v2.5 source data and organ-tree generation logic,
then adds the requested visible labels. Existing notebooks and SVG outputs are
not modified.


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import vl_convert as vlc
from collections import Counter
import json
import hashlib
from pathlib import Path
from IPython.display import display
import io
import re

pd.options.display.max_columns = None

In [ ]:
# nodes_url = "https://raw.githubusercontent.com/cns-iu/hra-ftu-vccf-supporting-information/main/code/data/asct-nodes.csv" 
# edges_url = "https://raw.githubusercontent.com/cns-iu/hra-ftu-vccf-supporting-information/main/code/data/asct-edges.csv"
# blood_edges_url = "https://raw.githubusercontent.com/cns-iu/hra-ftu-vccf-supporting-information/main/code/data/asct-blood-vasculature-edges.csv"

#use build_network.py to build data for latest release and save it in folder data
# data_version = 'v2.2'
# data_folder = f'All_Versions/data/data_{data_version}/'

# nodes_url = f"{data_folder}asct-nodes.csv" 
# edges_url = f"{data_folder}asct-edges.csv"
# blood_edges_url = f"{data_folder}asct-blood-vasculature-edges.csv"

# output_path = f'All_Versions/viz/viz_{data_version}/'

nodes_url = "data/v2.5/asct-nodes.csv" 
edges_url = "data/v2.5/asct-edges.csv"
# blood_edges_url = "data/asct-blood-vasculature-edges.csv"

# vessel_url = "data/Vessel.csv"
ftu_vessel_url = 'data/Table_S1.csv'

output_path = str(
    Path(
        "/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftu_horizontal_generator/highlighted_cell_labels_v4_red_nodes"
    )
)
print("output_path =", output_path)
## Make sure to save vega_config file in output_path/vega_config/

canvas_size = 1720

nodes = pd.read_csv(nodes_url)
edges = pd.read_csv(edges_url)

omitted_organs = ['muscular_system', 'skeleton', 'lymph_vasculature', 'peripheral_nervous_system', 'blood_pelvis']

organ_order = ['trachea', 'main bronchus', 'respiratory system', 'mouth', 'heart', 'spinal cord', 'brain', 'eye', 'palatine tonsil', 
               'skin of body', 'thymus', 'lymph node', 'spleen',  'liver', 'Pancreas', 'small intestine',  'large intestine', 
               'kidney', 'urinary bladder', 'ureter', 'prostate gland',  'ovary', 'fallopian tube', 'uterus',  
               'placenta', 'knee', 'Bone marrow']

nodes = nodes[~nodes['organ'].isin(omitted_organs)]
edges = edges[~edges['organ'].isin(omitted_organs)]

## Detach respiratory system nodes from body and attach to lung.
respiratory_system_nodes = ['UBERON:0008886', 'UBERON:0004573', 'UBERON:0003920']

new_edges = edges[(edges['source']=='UBERON:0013702')&(edges['target'].isin(respiratory_system_nodes))].copy()
#connect the nodes to respiratory system instead of body
new_edges['source'].replace('UBERON:0013702', 'UBERON:0001004', inplace=True) 

#Update the edges
edges = edges[~((edges['source']=='UBERON:0013702')&(edges['target'].isin(respiratory_system_nodes)))]
edges = pd.concat([edges, new_edges])

whole_graph = nx.from_pandas_edgelist(edges, source='source', target='target', edge_attr=True)
nx.is_tree(whole_graph)

def get_nodes(name=None, ontology_id=None):
    if name:
        return nodes[nodes['name'].str.lower()==name.lower()]
    if ontology_id:
        return nodes[nodes['ontology_id']==ontology_id]

# organ_order_in_id = [get_nodes(name=organ)['id'].values[0] for organ in organ_order]

organ_order_in_id = []

for organ in organ_order:
    if len(get_nodes(name=organ)) > 0:
        organ_order_in_id += (list(get_nodes(name=organ)['id'].values))

def get_organ_graph(organ_id):
    organ_graph = whole_graph.copy()
    organ_graph.remove_edge('UBERON:0013702', organ_id) #remove the edge between body and the organ

    connected_components = sorted(list(nx.connected_components(organ_graph)), key=len, reverse=False) #get the connected components

    return nx.subgraph(organ_graph, connected_components[0]) #return the smallest connected component

#List of ontology ids of the organs that are branching out from body
organs_origin_body_id = list(nodes[nodes['ontology_id'].isin(list(nx.neighbors(whole_graph, 'UBERON:0013702')))]['ontology_id'])

#Get the updated list of organ ids in order containing only ids that are connected to the body node.
new_organ_order_in_id = []
for organ_id in organ_order_in_id:
    if organ_id in organs_origin_body_id:
        new_organ_order_in_id.append(organ_id)

organ_order_in_id = new_organ_order_in_id

## Generate a new integer-based ID, that takes into account the organ order
#(This is necesary for the layout, later we can order the nodes and edges based on this integer ID, and then vega will visualize the organs in the desired order)

id_to_graph_int_id = {'UBERON:0013702': 0}

next_id = 1

for i, organ_id in enumerate(organ_order_in_id):
    branch_graph = get_organ_graph(organ_id)

    branch_graph_new_ids = dict(zip(nx.dfs_preorder_nodes(branch_graph, source=organ_id), range(next_id, next_id+branch_graph.number_of_nodes())))

    id_to_graph_int_id.update(branch_graph_new_ids)

    next_id += branch_graph.number_of_nodes()


nodes['graph_int_id'] = nodes['id'].map(id_to_graph_int_id)

edges['source_int'] = edges['source'].map(id_to_graph_int_id)
edges['target_int'] = edges['target'].map(id_to_graph_int_id)

whole_graph_int = nx.from_pandas_edgelist(edges, source='source_int', target='target_int', edge_attr=True)

graph_int_id_to_id = {v: k for k, v in id_to_graph_int_id.items()}

# remove those blood nodes from the graph, which if we remove, the graph will be still connected, repeat until no such node is found.
blood_nodes_candidates = set(nodes[nodes['organ']=='blood_vasculature'].graph_int_id.tolist())

distance_from_body = nx.single_source_shortest_path_length(whole_graph_int, source=0)

distance_from_body_blood_nodes = {k: v for k, v in distance_from_body.items() if k in blood_nodes_candidates}

node_visiting_order = sorted(distance_from_body_blood_nodes.items(), key=lambda x: x[1], reverse=True)

new_truncated_graph = whole_graph_int.copy()

for node in node_visiting_order:
    new_truncated_graph.remove_node(node[0])
    if nx.is_connected(new_truncated_graph):
        blood_nodes_candidates.remove(node[0])
    else:
        break

nodes['keep'] = nodes.apply(lambda row: True if (row['organ']!='blood_vasculature' or row['graph_int_id'] in blood_nodes_candidates) else False, axis=1)

truncated_nodes = nodes[nodes['keep']]
truncated_edges = edges[edges['source_int'].isin(truncated_nodes['graph_int_id']) | edges['target_int'].isin(truncated_nodes['graph_int_id'])]
truncated_graph = nx.from_pandas_edgelist(truncated_edges, source='source_int', target='target_int', edge_attr=True)

# ============================================================
# AUTHORITATIVE TABLE_S1 FTU INVENTORY
# ============================================================

table_s1_ftu = pd.read_csv(ftu_vessel_url)

TABLE_S1_FTU_DEFINITIONS = (
    table_s1_ftu[["FTU", "FTUID"]]
    .dropna(subset=["FTU", "FTUID"])
    .assign(
        FTU=lambda frame: (
            frame["FTU"].astype(str).str.strip()
        ),
        FTUID=lambda frame: (
            frame["FTUID"].astype(str).str.strip()
        ),
    )
    .drop_duplicates(subset=["FTU", "FTUID"])
    .sort_values(["FTU", "FTUID"])
    .reset_index(drop=True)
)

if TABLE_S1_FTU_DEFINITIONS["FTU"].nunique() != 28:
    raise ValueError("Expected 28 unique Table_S1 FTU names")

if TABLE_S1_FTU_DEFINITIONS["FTUID"].nunique() != 28:
    raise ValueError("Expected 28 unique Table_S1 FTU ontology IDs")

TABLE_S1_FTU_ID_BY_NAME = {
    str(row.FTU).strip().lower(): str(row.FTUID).strip()
    for row in TABLE_S1_FTU_DEFINITIONS.itertuples(index=False)
}

# Table_S1 uses the generic crypt ID UBERON:0001983, while the
# source graph represents the requested large-intestine subtype as
# UBERON:0001984. This is a source-established graph alias, not an
# invented FTU relationship.
TABLE_S1_GRAPH_FTU_ID_OVERRIDES = {
    "crypt of lieberkuhn": "UBERON:0001984",
}

FTUs = sorted(
    TABLE_S1_FTU_DEFINITIONS["FTUID"].unique().tolist()
)

CONFIRMED_GRAPH_FTU_IDS = (
    set(FTUs)
    | set(TABLE_S1_GRAPH_FTU_ID_OVERRIDES.values())
)

GRAPH_FTU_ID_TO_TABLE_S1_ID = {
    ftu_id: ftu_id
    for ftu_id in FTUs
}

for ftu_name, graph_ftu_id in (
    TABLE_S1_GRAPH_FTU_ID_OVERRIDES.items()
):
    GRAPH_FTU_ID_TO_TABLE_S1_ID[graph_ftu_id] = (
        TABLE_S1_FTU_ID_BY_NAME[ftu_name]
    )

print(
    "Table_S1 FTU inventory:",
    TABLE_S1_FTU_DEFINITIONS["FTU"].nunique(),
    "unique names /",
    TABLE_S1_FTU_DEFINITIONS["FTUID"].nunique(),
    "unique ontology IDs"
)


Main_Organs = list(nodes[nodes['ontology_id'].isin(list(nx.neighbors(whole_graph, 'UBERON:0013702')))]['id'])

ORGAN_FTU_NAMES = {
    "liver": {
        "liver lobule",
    },

    "kidney": {
        "cortical collecting duct",
        "descending limb of loop of henle",
        "inner medullary collecting duct",
        "loop of henle ascending limb thin segment",
        "nephron",
        "outer medullary collecting duct",
        "renal corpuscle",
        "thick ascending limb of loop of henle",
    },

    "large_intestine": {
        "crypt of lieberkuhn",
        "crypt of lieberkuhn of large intestine",
    },

    "small_intestine": {
        "intestinal villus",
    },

    "mouth": {
        "gingival attachment",
        "parotid gland ducto-acinar unit",
        "sublingual gland ducto-acinar unit",
        "submandibular gland ducto-acinar unit",
    },

    "prostate": {
        "prostate glandular acinus",
    },

    "lung": {
        "alveolus of lung",
        "bronchus submucosal gland",
    },

    "skin": {
        "dermal papilla",
        "epidermal ridge of digit",
        "hair follicle",
    },

    "brain": set(),
    "heart": set(),
}

In [ ]:
def construct_network_create_vega_viz(
    nodes_dataframe,
    edges_dataframe,
    filename="butterfly",
    root_id=0,
    only_female=False,
    only_male=False,
    scenegraph=False,
    show_labels=False
):
    """
    Construct an organ/tree Vega configuration.

    Colors:
      Red    = FTU
      Purple = Anatomical Structure (AS)
      Orange = Cell Type / other non-AS nodes

    Labels are controlled by the `show_label` column.
    """

    # -------------------------------------------------
    # Copy and sort input data
    # -------------------------------------------------
    nodes_df = (
        nodes_dataframe
        .sort_values("graph_int_id")
        .copy()
    )

    edges_df = (
        edges_dataframe
        .sort_values("target_int")
        .copy()
    )

    # -------------------------------------------------
    # Build graph
    # -------------------------------------------------
    graph = nx.from_pandas_edgelist(
        edges_df,
        source="source_int",
        target="target_int",
        edge_attr=True,
        create_using=nx.DiGraph()
    )

    # -------------------------------------------------
    # Determine parent of each node
    # -------------------------------------------------
    parents_dict = dict(
        nx.bfs_predecessors(
            graph,
            source=root_id
        )
    )

    nodes_df["parent"] = (
        nodes_df["graph_int_id"]
        .apply(
            lambda x: parents_dict.get(x, 0)
        )
        .astype(int)
    )

    # -------------------------------------------------
    # Normalize FTU IDs
    # -------------------------------------------------
    ftu_set = {
        str(x).strip()
        for x in CONFIRMED_GRAPH_FTU_IDS
    }

    ontology_ids = (
        nodes_df["ontology_id"]
        .astype(str)
        .str.strip()
    )

    # -------------------------------------------------
    # Node colors
    # -------------------------------------------------
    nodes_df["color"] = np.where(
        ontology_ids.isin(ftu_set),
        "#C62828",       # FTU = red
        np.where(
            nodes_df["type"].eq("AS"),
            "#984ea0",   # AS = purple
            "#ff7f00"    # CT / other = orange
        )
    )

    # -------------------------------------------------
    # Preserve explicit caller-provided label choices.
    # If no label column was supplied, show FTUs only
    # when show_labels=True.
    # -------------------------------------------------
    if "show_label" in nodes_df.columns:
        nodes_df["show_label"] = (
            pd.to_numeric(
                nodes_df["show_label"],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )
    elif show_labels:
        nodes_df["show_label"] = (
            ontology_ids.isin(ftu_set)
        ).astype(int)
    else:
        nodes_df["show_label"] = 0

    # -------------------------------------------------
    # Clean names
    # -------------------------------------------------
    nodes_df["name"] = (
        nodes_df["name"]
        .fillna("")
        .astype(str)
        .str.capitalize()
    )

    # -------------------------------------------------
    # Vega uses graph_int_id as node id
    # -------------------------------------------------
    nodes_df.rename(
        columns={
            "id": "id_from_ontology_id",
            "graph_int_id": "id"
        },
        inplace=True
    )

    # -------------------------------------------------
    # Scenegraph mode
    # -------------------------------------------------
    if scenegraph:

        nodes_df.rename(
            columns={"name": "name_label"},
            inplace=True
        )

        nodes_df["name"] = (
            nodes_df["id"].copy()
        )

    # -------------------------------------------------
    # Female / male filtering
    # -------------------------------------------------
    if only_female and not only_male:

        nodes_df = nodes_df[
            nodes_df["organ"] != "prostate"
        ]

        fn = filename + "_female"

    elif only_male and not only_female:

        nodes_df = nodes_df[
            ~nodes_df["organ"].isin(
                [
                    "fallopian_tube",
                    "ovary",
                    "uterus",
                    "placenta"
                ]
            )
        ]

        fn = filename + "_male"

    elif not only_female and not only_male:

        fn = filename + "_full"

    else:

        raise ValueError(
            "The parameters `only_female` and "
            "`only_male` cannot both be True."
        )

    # -------------------------------------------------
    # Convert nodes to Vega data
    # -------------------------------------------------
    nodes_json = nodes_df[
        [
            "id",
            "name",
            "parent",
            "type",
            "ontology_id",
            "id_from_ontology_id",
            "color",
            "organ",
            "show_label"
        ]
    ].to_dict(
        orient="index"
    )

    # -------------------------------------------------
    # Root must not have a parent
    # -------------------------------------------------
    for _, node in nodes_json.items():

        if int(node["id"]) == int(root_id):

            node.pop(
                "parent",
                None
            )

            break

    nodes_json = list(
        nodes_json.values()
    )

    # -------------------------------------------------
    # Load Vega template
    # -------------------------------------------------
    vega_config_path = (
        Path(output_path)
        / "vega_config"
        / "vega_config_horizontal.json"
    )

    if not vega_config_path.exists():

        raise FileNotFoundError(
            f"Vega template not found: "
            f"{vega_config_path}"
        )

    with open(
        vega_config_path,
        encoding="utf8"
    ) as json_file:

        config = json.load(
            json_file
        )

    # -------------------------------------------------
    # Use a compact canvas sized to the selected graph.
    # The tree stays near the upper-left with a small
    # margin; width follows graph depth and height follows
    # the number of terminal branches.
    # -------------------------------------------------
    depth_by_node = nx.single_source_shortest_path_length(
        graph,
        root_id
    )

    max_depth = max(
        (
            depth_by_node.get(int(node["id"]), 0)
            for node in nodes_json
        ),
        default=1
    )

    leaf_count = max(
        1,
        sum(
            graph.out_degree(int(node["id"])) == 0
            for node in nodes_json
        )
    )

    # Use a true compact hierarchy layout: every successive depth is
    # 100 px apart. Angled labels keep the tighter levels readable.
    # The right margin is reserved for readable labels; fonts are unchanged.
    level_spacing = 100
    tree_left_margin = 50
    tree_right_label_margin = 260
    compact_tree_width = level_spacing * max_depth

    config["width"] = int(
        max(
            800,
            min(2000, tree_left_margin + compact_tree_width + tree_right_label_margin)
        )
    )

    # Keep every organ within the landscape-page height used
    # successfully by Kidney. Dense unlabeled nodes may overlap;
    # the important Body/organ/FTU labels retain their spacing.
    base_tree_height = int(
        max(
            320,
            min(650, 100 + 22 * leaf_count)
        )
    )
    rotated_label_clearance = 180
    config["height"] = base_tree_height + rotated_label_clearance

    config["padding"] = 0

    tree_data = config["data"][0]

    for transform in tree_data.get("transform", []):

        if transform.get("type") == "tree":
            transform["size"] = [
                max(150, base_tree_height - 80),
                compact_tree_width
            ]

        elif transform.get("as") == "x_draw":
            transform["expr"] = f"datum.y + {tree_left_margin}"

        elif transform.get("as") == "y_draw":
            transform["expr"] = "datum.x + 220"

    for mark in config.get("marks", []):

        if mark.get("type") == "path":
            mark["encode"]["update"]["strokeWidth"] = {
                "value": 1.1
            }

        elif mark.get("type") == "symbol":
            mark["encode"]["enter"]["size"] = {
                "value": 45
            }
            mark["encode"]["enter"]["strokeWidth"] = {
                "value": 0.5
            }

        elif mark.get("type") == "text":
            mark["encode"]["enter"]["fontSize"] = {
                "value": 12
            }

    # -------------------------------------------------
    # Insert data
    # -------------------------------------------------
    config["data"][0]["values"] = nodes_json

    # -------------------------------------------------
    # IMPORTANT:
    # Vega displays only nodes with show_label == 1
    # -------------------------------------------------
    text_mark = config["marks"][-1]

    text_mark["encode"]["update"]["opacity"] = {
        "signal": "datum.show_label == 1 ? 1 : 0"
    }

    # Small gap between node and label
    text_mark["encode"]["update"]["dx"] = {
        "value": 7
    }

    text_mark["encode"]["update"]["dy"] = {
        "value": 0
    }

    text_mark["encode"]["update"]["align"] = {
        "value": "left"
    }

    text_mark["encode"]["update"]["baseline"] = {
        "value": "middle"
    }

    # -------------------------------------------------
    # Save Vega JSON
    # -------------------------------------------------
    output_file = (
        Path(output_path)
        / f"{fn}_vega_viz_config.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf8"
    ) as outfile:

        json.dump(
            config,
            outfile,
            indent=4
        )

    print(
        f'File saved as "{output_file}"'
    )

    return config

In [ ]:
def create_vega_viz(config, filename, scenegraph=False):
    '''
    Create the visualization based on the vega config file. The visualization can be saved as SVG or as a scenegraph JSON file.

    Parameters:
    config: dict
        The vega config file.
    filename: str
        The name of the file to be saved.
    scenegraph: bool
        If True, the scenegraph JSON file will be saved, otherwise the SVG file will be saved.
    '''

    if scenegraph:
        scenegraph_data = vlc.vega_to_scenegraph(vg_spec=config)

        with open(f"{output_path}/vega_{filename}_scenegraph.json", "w") as outfile:
            outfile.write(json.dumps(scenegraph_data, indent=4))

        return scenegraph_data
    
    else:
        svg_str = vlc.vega_to_svg(vg_spec=config,)

        with open(f"{output_path}/vega_{filename}_viz.svg", "wt") as f:
            f.write(svg_str)

        return svg_str

In [ ]:
# ============================================================
# FINAL ORGAN-SPECIFIC TREE GENERATION
# ============================================================

# ------------------------------------------------------------
# Requested organs
# ------------------------------------------------------------

organ_specs = [
    {"organ": "brain",           "root_id": 5605},
    {"organ": "heart",           "root_id": 380},
    {"organ": "large_intestine", "root_id": 7661},
    {"organ": "small_intestine", "root_id": 7473},
    {"organ": "liver",           "root_id": 7246},
    {"organ": "prostate",        "root_id": 9201},
    {"organ": "kidney",          "root_id": 8991},
    {"organ": "mouth",           "root_id": 277},
    {"organ": "lung",            "root_id": 164},
    {"organ": "skin",            "root_id": 6719},
]

# Exact baseline fingerprints from the current validated organ trees.
# Each digest covers graph ID, parent edge, name, node type, ontology ID,
# color, and organ. Layout-only edits must leave every digest unchanged.
EXPECTED_TREE_TOPOLOGY_SHA256 = {
    "lung": "9678ee3adc7e682be87de9effba88920134d55a8a40d076a0832aa6e1bcb5a22",
    "prostate": "6fa866e8ce595f3e915589468e506ff7acfc519438e7216169457035b9421510",
    "skin": "b1c89cbe6dbec89ae300a763e5d88acef82be3ec1eca8fc73e658ac0f99ef675",
    "small_intestine": "1f7f538c03186989e752fc3f55ed657dea006b0bb704d6c52432b583418656b0",
    "large_intestine": "739c36a506952ee65903154d12b0e3ee35e68e33e468c98f41bdde4a927d9d6e",
    "liver": "6c049dd1e85687e4e9f44b428028fa49c4f04913e736953ed061a0703df00dc4",
    "kidney": "f2bc8a3578017f752205e8a54b2b0e105419f314565f78865868c2ac55570b45",
    "brain": "9f2f0435cba30bfcc9d63006c725409727d60cbfdf4cec312c519f646ea8490b",
    "heart": "356e95df2ad10a854031200200332874b14056e231f1fd3c6fa122df25e756c9",
    "mouth": "46e9a9aa6526da8a1de68d445ea1d86ca595a3975f70129c63b25a9c61c3a9e3",
}


# ------------------------------------------------------------
# Table_S1-derived FTUs to display for each requested organ
#
# Table_S1 provides FTU names and ontology IDs but no organ column.
# Organ ownership comes from the curated ORGAN_FTU_NAMES definitions;
# PathVessel and the node organ metadata column are not used.
# ------------------------------------------------------------

ORGAN_FTU_IDS = {}

for organ_spec in organ_specs:
    organ_name = organ_spec["organ"]
    resolved_ids = set()

    for configured_ftu_name in ORGAN_FTU_NAMES.get(
        organ_name,
        set()
    ):
        normalized_ftu_name = (
            str(configured_ftu_name)
            .strip()
            .lower()
        )

        table_s1_ftu_id = TABLE_S1_FTU_ID_BY_NAME.get(
            normalized_ftu_name
        )

        if table_s1_ftu_id is None:
            continue

        resolved_ids.add(
            TABLE_S1_GRAPH_FTU_ID_OVERRIDES.get(
                normalized_ftu_name,
                table_s1_ftu_id
            )
        )

    ORGAN_FTU_IDS[organ_name] = resolved_ids

# Brain and Heart are explicitly non-FTU for the current Table_S1
# inventory. Do not infer or invent FTUs for them.
assert ORGAN_FTU_IDS["brain"] == set()
assert ORGAN_FTU_IDS["heart"] == set()


# Preferred display text for FTUs whose source name is blank or broader.
FTU_LABEL_OVERRIDES = {
    "UBERON:8000011": "Parotid gland ducto-acinar unit",
    "UBERON:8000012": "Sublingual gland ducto-acinar unit",
    "UBERON:8000013": "Submandibular gland ducto-acinar unit",
}


# ------------------------------------------------------------
# Editorial label targets requested for the revised organ trees
# ------------------------------------------------------------

HIGHLIGHT_TARGETS = {
    "prostate": [
        {
            "key": "prostatic_urethra",
            "ontology_id": "UBERON:0001335",
            "label": "Prostatic urethra",
        },
        {
            "key": "hillock_cell",
            "ontology_id": "CL:0017011",
            "label": "Hillock cell of the prostatic urethral epithelium",
            "parent_key": "prostatic_urethra",
        },
        {
            "key": "prostate_acinus",
            "ontology_id": "UBERON:0004179",
            "label": "Prostate glandular acinus",
        },
        {
            "key": "prostate_luminal_cell",
            "ontology_id": "CL:0002340",
            "label": "Luminal cell of the prostate epithelium",
            "parent_key": "prostate_acinus",
        },
    ],
    "lung": [
        {
            "key": "alveolus",
            "ontology_id": "UBERON:0002299",
            "label": "Alveolus of lung",
        },
        {
            "key": "type_i_pneumocyte",
            "ontology_id": "CL:0002062",
            "label": "Type I pneumocyte",
            "parent_key": "alveolus",
        },
        {
            "key": "bronchus_submucosal_gland",
            "ontology_id": "UBERON:8410043",
            "label": "Bronchial submucosal gland",
        },
        {
            "key": "type_ii_pneumocyte",
            "ontology_id": "CL:0002063",
            "label": "Type II pneumocyte",
            "parent_key": "alveolus",
        },
    ],
    "brain": [
        {
            "key": "amygdala",
            "ontology_id": "UBERON:0001876",
            "label": "Amygdala",
        },
        {
            "key": "hypothalamus",
            "ontology_id": "UBERON:0001898",
            "label": "Hypothalamus",
        },
        {
            "key": "pineal_gland",
            "ontology_id": "UBERON:0001905",
            "label": "Pineal gland",
        },
    ],
    "heart": [
        {
            "key": "cardiomyocyte",
            "ontology_id": "CL:0000746",
            "label": "Cardiomyocyte",
        },
    ],
    "kidney": [
        {
            "key": "renal_corpuscle",
            "ontology_id": "UBERON:0001229",
            "label": "Renal corpuscle",
        },
        {
            "key": "podocyte",
            "ontology_id": "CL:0000653",
            "label": "Podocyte",
            "parent_key": "renal_corpuscle",
        },
    ],
    "liver": [
        {
            "key": "liver_lobule",
            "ontology_id": "UBERON:0004647",
            "label": "Hepatic lobule",
        },
        {
            "key": "kupffer_cell",
            "ontology_id": "CL:0000091",
            "label": "Kupffer cell",
            "parent_key": "liver_lobule",
        },
    ],
}

# These requested brain terms do not occur as nodes in the HRA v2.5
# ASCT+B source used by the original notebook. They are shown in a
# separate editorial annotation block so the source hierarchy is not
# altered or misrepresented.
BRAIN_EDITORIAL_ANNOTATIONS = [
    "Amygdala (AS): pyramidal glutamatergic neurons (CT)",
    "Hypothalamus (AS): magnocellular neurosecretory cells (CT)",
    "Pituitary gland (AS): somatotrophs (CT)",
    "Pineal gland (AS): pinealocytes (CT)",
]


# ------------------------------------------------------------
# Build the full directed graph ONCE
# ------------------------------------------------------------

graph_edges = truncated_edges.dropna(
    subset=["source_int", "target_int"]
).copy()

graph_edges["source_int"] = pd.to_numeric(
    graph_edges["source_int"],
    errors="coerce"
)

graph_edges["target_int"] = pd.to_numeric(
    graph_edges["target_int"],
    errors="coerce"
)

graph_edges = graph_edges.dropna(
    subset=["source_int", "target_int"]
).copy()

graph_edges["source_int"] = (
    graph_edges["source_int"].astype(int)
)

graph_edges["target_int"] = (
    graph_edges["target_int"].astype(int)
)

full_graph = nx.from_pandas_edgelist(
    graph_edges,
    source="source_int",
    target="target_int",
    create_using=nx.DiGraph()
)

body_root_id = 0


# ------------------------------------------------------------
# Graph-node metadata lookups
#
# These use graph IDs only. The organ metadata column is not
# used to determine subtree membership.
# ------------------------------------------------------------

node_metadata = truncated_nodes.copy()

node_metadata["graph_int_id"] = pd.to_numeric(
    node_metadata["graph_int_id"],
    errors="coerce"
)

node_metadata = (
    node_metadata
    .dropna(subset=["graph_int_id"])
    .copy()
)

node_metadata["graph_int_id"] = (
    node_metadata["graph_int_id"].astype(int)
)

node_metadata = (
    node_metadata
    .drop_duplicates(subset=["graph_int_id"], keep="first")
    .copy()
)

node_type_by_id = dict(
    zip(
        node_metadata["graph_int_id"],
        node_metadata["type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )
)

node_ontology_by_id = dict(
    zip(
        node_metadata["graph_int_id"],
        node_metadata["ontology_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
)

global_ftu_ids = {
    str(value).strip()
    for value in CONFIRMED_GRAPH_FTU_IDS
}


# ============================================================
# GENERATE EACH ORGAN
# ============================================================

for spec in organ_specs:

    organ = spec["organ"]
    organ_root_id = int(spec["root_id"])

    print("\n" + "=" * 75)
    print("BUILDING:", organ.upper())
    print("ROOT ID:", organ_root_id)
    print("=" * 75)

    # --------------------------------------------------------
    # Check organ root
    # --------------------------------------------------------

    if organ_root_id not in full_graph:

        print(
            f"SKIPPING {organ}: "
            f"root {organ_root_id} not found"
        )

        continue

    # --------------------------------------------------------
    # Body -> Organ path
    # --------------------------------------------------------

    try:

        body_to_organ = nx.shortest_path(
            full_graph,
            source=body_root_id,
            target=organ_root_id
        )

    except nx.NetworkXNoPath:

        print(
            f"SKIPPING {organ}: "
            "no path from Body to organ"
        )

        continue

    # --------------------------------------------------------
    # All nodes underneath this organ
    # --------------------------------------------------------

    organ_descendants = nx.descendants(
        full_graph,
        organ_root_id
    )

    organ_reachable_ids = (
        set(organ_descendants)
        | {organ_root_id}
    )

    # --------------------------------------------------------
    # FTUs requested for this organ
    # --------------------------------------------------------

    desired_ftu_ids = {
        str(x).strip()
        for x in ORGAN_FTU_IDS.get(
            organ,
            set()
        )
    }

    print(
        "Requested FTU IDs:",
        desired_ftu_ids
    )

    # --------------------------------------------------------
    # Find ONE graph node for each FTU ID
    # --------------------------------------------------------

    representative_ftu_nodes = {}

    for ftu_id in sorted(desired_ftu_ids):

        candidates_df = truncated_nodes[
            truncated_nodes["ontology_id"]
            .astype(str)
            .str.strip()
            .eq(ftu_id)
        ].copy()

        # ----------------------------------------------------
        # Safe numeric graph IDs
        # ----------------------------------------------------

        candidates_df["graph_int_id"] = pd.to_numeric(
            candidates_df["graph_int_id"],
            errors="coerce"
        )

        candidates_df = candidates_df.dropna(
            subset=["graph_int_id"]
        ).copy()

        candidates_df["graph_int_id"] = (
            candidates_df["graph_int_id"]
            .astype(int)
        )

        # ----------------------------------------------------
        # Remove duplicate dataframe rows
        # ----------------------------------------------------

        candidates_df = (
            candidates_df
            .drop_duplicates(
                subset=["graph_int_id"],
                keep="first"
            )
            .sort_values("graph_int_id")
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Only candidates actually below this organ
        # ----------------------------------------------------

        candidates_df = candidates_df[
            candidates_df["graph_int_id"]
            .isin(organ_reachable_ids)
        ].copy()

        # ----------------------------------------------------
        # For Lung, explicitly use the alveolus name
        # ----------------------------------------------------

        if (
            organ == "lung"
            and ftu_id == "UBERON:0002299"
        ):

            named_candidates = candidates_df[
                candidates_df["name"]
                .astype(str)
                .str.strip()
                .str.lower()
                .eq("alveolus of lung")
            ].copy()

            if len(named_candidates) > 0:
                candidates_df = named_candidates

        # ----------------------------------------------------
        # Pick the FIRST reachable candidate
        # ----------------------------------------------------

        candidates = (
            candidates_df["graph_int_id"]
            .astype(int)
            .tolist()
        )

        for candidate in candidates:

            candidate = int(candidate)

            if candidate not in full_graph:
                continue

            try:
                nx.shortest_path(
                    full_graph,
                    source=organ_root_id,
                    target=candidate
                )

                representative_ftu_nodes[
                    ftu_id
                ] = candidate

                break

            except nx.NetworkXNoPath:
                continue

    print(
        "Representative FTUs:",
        representative_ftu_nodes
    )

    missing_ftu_ids = (
        desired_ftu_ids
        - set(representative_ftu_nodes)
    )

    if missing_ftu_ids:
        print(
            "WARNING - requested FTUs not reachable from",
            organ_root_id,
            ":",
            sorted(missing_ftu_ids)
        )

    # --------------------------------------------------------
    # Build ONLY the selected tree
    # --------------------------------------------------------

    selected_node_ids = {
        int(x)
        for x in body_to_organ
    }

    selected_node_ids.add(
        organ_root_id
    )

    # --------------------------------------------------------
    # FTU organs:
    #   organ -> one representative FTU -> CT/other layer
    #
    # Non-FTU organs:
    #   organ -> first-level AS -> second-level AS -> direct
    #   CT/other children. This is a controlled structural
    #   view, not the unrestricted full descendant subtree.
    # --------------------------------------------------------

    first_level_as_ids = set()
    second_level_as_ids = set()
    selected_structural_ct_ids = set()

    if desired_ftu_ids:

        for ftu_id, ftu_node in (
            representative_ftu_nodes.items()
        ):

            path_to_ftu = nx.shortest_path(
                full_graph,
                source=organ_root_id,
                target=ftu_node
            )

            selected_node_ids.update(
                int(x)
                for x in path_to_ftu
            )

            # Traverse the representative FTU branch using
            # source edges. Stop at any additional FTU node so
            # duplicate FTU occurrences and their branches are
            # not reintroduced below the chosen representative.
            descendant_stack = [
                int(node_id)
                for node_id in full_graph.successors(
                    ftu_node
                )
            ]

            visited_descendants = set()

            while descendant_stack:
                descendant_id = int(
                    descendant_stack.pop()
                )

                if descendant_id in visited_descendants:
                    continue

                visited_descendants.add(
                    descendant_id
                )

                if node_ontology_by_id.get(
                    descendant_id,
                    ""
                ) in global_ftu_ids:
                    continue

                selected_node_ids.add(
                    descendant_id
                )

                descendant_stack.extend(
                    int(node_id)
                    for node_id in full_graph.successors(
                        descendant_id
                    )
                )

    else:

        # ----------------------------------------------------
        # Candidate AS hierarchy diagnostic BEFORE pruning.
        #
        # Print every AS candidate grouped by directed depth
        # from the organ root. The retained levels below are
        # therefore explicit and graph-derived, never selected
        # by dataframe order, head(n), or visual appearance.
        # ----------------------------------------------------

        organ_depth_by_id = (
            nx.single_source_shortest_path_length(
                full_graph,
                organ_root_id
            )
        )

        candidate_as_ids = {
            int(node_id)
            for node_id, depth in organ_depth_by_id.items()
            if node_type_by_id.get(
                int(node_id),
                ""
            ) == "AS"
        }

        print(
            "\nCANDIDATE AS HIERARCHY BEFORE PRUNING:",
            organ
        )

        for candidate_depth in sorted(
            {
                int(organ_depth_by_id[node_id])
                for node_id in candidate_as_ids
            }
        ):
            print("  DEPTH", candidate_depth)

            for candidate_as_id in sorted(
                node_id
                for node_id in candidate_as_ids
                if int(
                    organ_depth_by_id[node_id]
                ) == candidate_depth
            ):
                metadata_rows = node_metadata[
                    node_metadata["graph_int_id"]
                    == candidate_as_id
                ]

                if len(metadata_rows) > 0:
                    metadata_row = metadata_rows.iloc[0]
                    candidate_name = str(
                        metadata_row["name"]
                    )
                    candidate_ontology_id = str(
                        metadata_row["ontology_id"]
                    )
                else:
                    candidate_name = ""
                    candidate_ontology_id = ""

                direct_as_children = sum(
                    node_type_by_id.get(
                        int(child_id),
                        ""
                    ) == "AS"
                    for child_id in full_graph.successors(
                        candidate_as_id
                    )
                )

                direct_ct_children = sum(
                    node_type_by_id.get(
                        int(child_id),
                        ""
                    ) != "AS"
                    and node_ontology_by_id.get(
                        int(child_id),
                        ""
                    ) not in global_ftu_ids
                    for child_id in full_graph.successors(
                        candidate_as_id
                    )
                )

                print(
                    {
                        "graph_int_id": candidate_as_id,
                        "name": candidate_name,
                        "ontology_id": candidate_ontology_id,
                        "depth": candidate_depth,
                        "direct_as_children": int(
                            direct_as_children
                        ),
                        "direct_ct_children": int(
                            direct_ct_children
                        ),
                    }
                )

        print(
            "RETAINED NON-FTU STRUCTURAL LEVELS:",
            "AS depth 1 and depth 2, plus their direct",
            "non-AS/non-FTU children"
        )

        first_level_as_ids = {
            int(node_id)
            for node_id in full_graph.successors(
                organ_root_id
            )
            if node_type_by_id.get(
                int(node_id),
                ""
            ) == "AS"
        }

        second_level_as_ids = {
            int(node_id)
            for first_level_id in first_level_as_ids
            for node_id in full_graph.successors(
                first_level_id
            )
            if node_type_by_id.get(
                int(node_id),
                ""
            ) == "AS"
        }

        selected_as_ids = (
            first_level_as_ids
            | second_level_as_ids
        )

        selected_node_ids.update(
            selected_as_ids
        )

        selected_structural_ct_ids = {
            int(node_id)
            for as_node_id in selected_as_ids
            for node_id in full_graph.successors(
                as_node_id
            )
            if node_type_by_id.get(
                int(node_id),
                ""
            ) != "AS"
            and node_ontology_by_id.get(
                int(node_id),
                ""
            ) not in global_ftu_ids
        }

        selected_node_ids.update(
            selected_structural_ct_ids
        )


    # --------------------------------------------------------
    # Add the requested editorial targets and their true paths
    # from the organ root. Candidate selection remains graph-
    # based and prefers the requested parent branch when given.
    # --------------------------------------------------------

    highlight_label_ids = set()
    highlight_label_text_by_id = {}
    selected_highlight_nodes = {}

    for target in HIGHLIGHT_TARGETS.get(organ, []):
        target_id = str(target["ontology_id"]).strip()
        target_rows = node_metadata[
            node_metadata["ontology_id"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq(target_id)
        ].copy()

        candidate_ids = [
            int(value)
            for value in target_rows["graph_int_id"].tolist()
            if int(value) in full_graph
        ]

        # Reuse the already selected FTU representative when the
        # requested label refers to that same FTU ontology ID.
        if target_id in representative_ftu_nodes:
            candidate_ids = [
                int(representative_ftu_nodes[target_id])
            ]

        parent_key = target.get("parent_key")
        preferred_source = selected_highlight_nodes.get(
            parent_key,
            organ_root_id,
        )

        reachable_candidates = []
        for candidate_id in candidate_ids:
            if nx.has_path(full_graph, preferred_source, candidate_id):
                reachable_candidates.append(
                    (
                        nx.shortest_path_length(
                            full_graph,
                            preferred_source,
                            candidate_id,
                        ),
                        candidate_id,
                    )
                )

        if not reachable_candidates and preferred_source != organ_root_id:
            for candidate_id in candidate_ids:
                if nx.has_path(full_graph, organ_root_id, candidate_id):
                    reachable_candidates.append(
                        (
                            nx.shortest_path_length(
                                full_graph,
                                organ_root_id,
                                candidate_id,
                            ),
                            candidate_id,
                        )
                    )
            preferred_source = organ_root_id

        # Some lung FTUs, especially bronchus submucosal gland,
        # occur on a respiratory branch that is connected to Body
        # but not below the selected Lung root. Retain the real
        # Body-derived path instead of fabricating an organ edge.
        if not reachable_candidates:
            for candidate_id in candidate_ids:
                if nx.has_path(full_graph, body_root_id, candidate_id):
                    reachable_candidates.append(
                        (
                            nx.shortest_path_length(
                                full_graph,
                                body_root_id,
                                candidate_id,
                            ),
                            candidate_id,
                        )
                    )
            preferred_source = body_root_id

        if not reachable_candidates:
            print(
                "HIGHLIGHT TARGET NOT FOUND:",
                organ,
                target["label"],
                target_id,
            )
            continue

        _, selected_target_id = min(reachable_candidates)
        selected_highlight_nodes[target["key"]] = selected_target_id
        target_path = nx.shortest_path(
            full_graph,
            preferred_source,
            selected_target_id,
        )
        selected_node_ids.update(int(value) for value in target_path)
        highlight_label_ids.add(selected_target_id)
        highlight_label_text_by_id[selected_target_id] = target["label"]

        print(
            "HIGHLIGHT TARGET:",
            organ,
            target["label"],
            "-> graph node",
            selected_target_id,
        )

    # --------------------------------------------------------
    # Extract selected nodes
    # --------------------------------------------------------

    node_graph_ids = pd.to_numeric(
        truncated_nodes["graph_int_id"],
        errors="coerce"
    )

    organ_nodes = truncated_nodes[
        node_graph_ids.isin(
            selected_node_ids
        )
    ].copy()

    # --------------------------------------------------------
    # Exactly ONE dataframe row per graph node
    # --------------------------------------------------------

    organ_nodes["graph_int_id"] = pd.to_numeric(
        organ_nodes["graph_int_id"],
        errors="coerce"
    )

    organ_nodes = organ_nodes.dropna(
        subset=["graph_int_id"]
    ).copy()

    organ_nodes["graph_int_id"] = (
        organ_nodes["graph_int_id"]
        .astype(int)
    )

    organ_nodes = (
        organ_nodes
        .drop_duplicates(
            subset=["graph_int_id"],
            keep="first"
        )
        .copy()
    )

    # --------------------------------------------------------
    # Extract selected edges
    # --------------------------------------------------------

    source_ids = pd.to_numeric(
        truncated_edges["source_int"],
        errors="coerce"
    )

    target_ids = pd.to_numeric(
        truncated_edges["target_int"],
        errors="coerce"
    )

    organ_edges = truncated_edges[
        source_ids.isin(selected_node_ids)
        &
        target_ids.isin(selected_node_ids)
    ].copy()

    # Remove duplicate edges

    organ_edges = (
        organ_edges
        .drop_duplicates(
            subset=[
                "source_int",
                "target_int"
            ],
            keep="first"
        )
        .copy()
    )

    # --------------------------------------------------------
    # Labels
    #
    # Label every node on the directed Body -> organ -> displayed FTU
    # paths. Nodes outside those paths remain intentionally unlabeled.
    # --------------------------------------------------------

    label_paths = [
        [int(node_id) for node_id in body_to_organ]
    ]

    for ftu_node_id in sorted(
        representative_ftu_nodes.values()
    ):
        label_paths.append(
            [
                int(node_id)
                for node_id in nx.shortest_path(
                    full_graph,
                    source=body_root_id,
                    target=int(ftu_node_id)
                )
            ]
        )

    label_ids = {
        node_id
        for label_path in label_paths
        for node_id in label_path
    }

    label_ids.update(highlight_label_ids)

    path_label_depth = {
        node_id: nx.shortest_path_length(
            full_graph,
            source=body_root_id,
            target=node_id
        )
        for node_id in label_ids
        if node_id in full_graph
        and nx.has_path(full_graph, body_root_id, node_id)
    }

    organ_nodes["show_label"] = (
        organ_nodes["graph_int_id"]
        .isin(label_ids)
        .astype(int)
    )

    # --------------------------------------------------------
    # Validation report (printed before Vega/SVG rendering)
    # --------------------------------------------------------

    selected_ontology_ids = (
        organ_nodes["ontology_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    selected_types = (
        organ_nodes["type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    selected_ftu_mask = (
        selected_ontology_ids.isin(global_ftu_ids)
    )

    selected_as_mask = (
        selected_types.eq("AS")
        & ~selected_ftu_mask
    )

    selected_ct_other_mask = ~(
        selected_ftu_mask
        | selected_as_mask
    )

    selected_ftu_report = []

    for ftu_id, graph_id in sorted(
        representative_ftu_nodes.items()
    ):
        selected_ftu_report.append(
            {
                "ontology_id": ftu_id,
                "graph_id": int(graph_id),
            }
        )

    body_to_organ_report = [
        int(node_id)
        for node_id in body_to_organ
    ]

    validation_report = {
        "organ": organ,
        "root_id": organ_root_id,
        "body_to_organ_path": body_to_organ_report,
        "as_nodes": int(selected_as_mask.sum()),
        "ct_other_nodes": int(
            selected_ct_other_mask.sum()
        ),
        "selected_nodes": len(organ_nodes),
        "selected_edges": len(organ_edges),
        "ftu_nodes": int(selected_ftu_mask.sum()),
        "unique_ftu_ontology_ids": int(
            selected_ontology_ids[
                selected_ftu_mask
            ].nunique()
        ),
        "selected_ftus": selected_ftu_report,
    }

    print("\nVALIDATION REPORT:")

    for key, value in validation_report.items():
        print(f"  {key}: {value}")

    print(
        "VALIDATION SUMMARY | "
        "organ | selected nodes | AS | FTU | CT/other | "
        "unique FTU IDs"
    )

    print(
        "VALIDATION SUMMARY |",
        organ,
        "|",
        len(organ_nodes),
        "|",
        int(selected_as_mask.sum()),
        "|",
        int(selected_ftu_mask.sum()),
        "|",
        int(selected_ct_other_mask.sum()),
        "|",
        int(
            selected_ontology_ids[
                selected_ftu_mask
            ].nunique()
        )
    )

    selected_graph = full_graph.subgraph(
        selected_node_ids
    )

    # --------------------------------------------------------
    # Per-representative FTU validation
    # --------------------------------------------------------

    for ftu_id, graph_id in sorted(
        representative_ftu_nodes.items()
    ):
        graph_id = int(graph_id)

        representative_rows = organ_nodes[
            organ_nodes["graph_int_id"] == graph_id
        ]

        source_name = (
            str(representative_rows.iloc[0]["name"])
            .strip()
            if len(representative_rows) > 0
            else ""
        )

        ftu_name = FTU_LABEL_OVERRIDES.get(
            ftu_id,
            source_name
        )

        representative_path = nx.shortest_path(
            full_graph,
            source=organ_root_id,
            target=graph_id
        )

        representative_descendants = nx.descendants(
            selected_graph,
            graph_id
        )

        representative_ct_descendants = sum(
            node_type_by_id.get(
                int(node_id),
                ""
            ) != "AS"
            and node_ontology_by_id.get(
                int(node_id),
                ""
            ) not in global_ftu_ids
            for node_id in representative_descendants
        )

        print(
            "FTU VALIDATION |",
            ftu_name,
            "|",
            "| Table_S1 ID:",
            GRAPH_FTU_ID_TO_TABLE_S1_ID.get(
                ftu_id,
                ftu_id
            ),
            "| graph FTU ID:",
            ftu_id,
            "| representative graph ID:",
            graph_id,
            "| path length:",
            len(representative_path) - 1,
            "| CT descendants:",
            int(representative_ct_descendants)
        )

    # --------------------------------------------------------
    # Lung: show every configured FTU candidate and whether it
    # is actually reachable from Lung root 164.
    # --------------------------------------------------------

    if organ == "lung":

        print("\nLUNG FTU CANDIDATE VALIDATION:")

        for ftu_id in sorted(desired_ftu_ids):

            all_candidate_rows = node_metadata[
                node_metadata["ontology_id"]
                .fillna("")
                .astype(str)
                .str.strip()
                .eq(ftu_id)
            ].sort_values("graph_int_id")

            all_candidate_ids = [
                int(value)
                for value in all_candidate_rows[
                    "graph_int_id"
                ].tolist()
            ]

            reachable_candidate_ids = [
                graph_id
                for graph_id in all_candidate_ids
                if graph_id in full_graph
                and nx.has_path(
                    full_graph,
                    organ_root_id,
                    graph_id
                )
            ]

            selected_representative = (
                representative_ftu_nodes.get(ftu_id)
            )

            if selected_representative is not None:
                selected_representative = int(
                    selected_representative
                )

                representative_path = nx.shortest_path(
                    full_graph,
                    source=organ_root_id,
                    target=selected_representative
                )

                retained_descendants = len(
                    nx.descendants(
                        selected_graph,
                        selected_representative
                    )
                )
            else:
                representative_path = None
                retained_descendants = 0

            source_name = (
                str(all_candidate_rows.iloc[0]["name"])
                .strip()
                if len(all_candidate_rows) > 0
                else ""
            )

            print(
                {
                    "ftu_name": FTU_LABEL_OVERRIDES.get(
                        ftu_id,
                        source_name
                    ),
                    "ontology_id": ftu_id,
                    "all_candidate_graph_ids": all_candidate_ids,
                    "reachable_from_lung_root_164": (
                        reachable_candidate_ids
                    ),
                    "selected_representative_graph_id": (
                        selected_representative
                    ),
                    "path_from_lung_root": representative_path,
                    "retained_descendants": retained_descendants,
                }
            )

    # --------------------------------------------------------
    # Prostate: print the exact Body -> representative FTU path
    # plus the retained orange CT/other descendants.
    # --------------------------------------------------------

    if organ == "prostate":

        print("\nPROSTATE PATH VALIDATION:")

        for graph_id in sorted(
            representative_ftu_nodes.values()
        ):
            prostate_path = nx.shortest_path(
                full_graph,
                source=body_root_id,
                target=int(graph_id)
            )

            for path_node_id in prostate_path:
                path_rows = organ_nodes[
                    organ_nodes["graph_int_id"]
                    == int(path_node_id)
                ]

                if len(path_rows) > 0:
                    path_row = path_rows.iloc[0]

                    print(
                        {
                            "graph_id": int(path_node_id),
                            "name": str(path_row["name"]),
                            "type": str(path_row["type"]),
                            "ontology_id": str(
                                path_row["ontology_id"]
                            ),
                        }
                    )

        print("PROSTATE ORANGE DESCENDANTS:")

        for _, orange_row in organ_nodes[
            selected_ct_other_mask
        ].sort_values("graph_int_id").iterrows():
            print(
                {
                    "graph_id": int(
                        orange_row["graph_int_id"]
                    ),
                    "name": str(orange_row["name"]),
                    "type": str(orange_row["type"]),
                    "ontology_id": str(
                        orange_row["ontology_id"]
                    ),
                }
            )

    # --------------------------------------------------------
    # Brain/Heart: exact deterministic structural rule and one
    # row for every selected AS node, including Body and root.
    # --------------------------------------------------------

    if organ in {"brain", "heart"}:

        print(
            "NON-FTU SELECTION RULE: Body -> organ; direct AS "
            "children of the organ root; direct AS children "
            "of those first-level AS nodes; direct non-AS, "
            "non-FTU children of the selected AS nodes. "
            "Membership and edges use only the directed graph."
        )

        print(
            "  ftu_count_confirmed_zero:",
            int(selected_ftu_mask.sum()) == 0
        )

        selected_id_set = set(
            organ_nodes["graph_int_id"].astype(int)
        )

        selected_as_table = []

        for _, as_row in organ_nodes[
            selected_as_mask
        ].sort_values("graph_int_id").iterrows():

            as_graph_id = int(
                as_row["graph_int_id"]
            )

            if as_graph_id == body_root_id:
                depth_from_organ_root = None
            else:
                depth_from_organ_root = (
                    nx.shortest_path_length(
                        full_graph,
                        source=organ_root_id,
                        target=as_graph_id
                    )
                )

            selected_ct_children = sum(
                int(child_id) in selected_id_set
                and node_type_by_id.get(
                    int(child_id),
                    ""
                ) != "AS"
                and node_ontology_by_id.get(
                    int(child_id),
                    ""
                ) not in global_ftu_ids
                for child_id in full_graph.successors(
                    as_graph_id
                )
            )

            selected_as_table.append(
                {
                    "graph_id": as_graph_id,
                    "name": str(as_row["name"]),
                    "ontology_id": str(
                        as_row["ontology_id"]
                    ),
                    "depth_from_organ_root": (
                        depth_from_organ_root
                    ),
                    "selected_ct_children": int(
                        selected_ct_children
                    ),
                }
            )

        print("SELECTED AS TABLE:")

        for selected_as_row in selected_as_table:
            print(selected_as_row)

        print(
            "  selected_ct_descendants:",
            len(selected_structural_ct_ids)
        )

    # --------------------------------------------------------
    # Build Vega configuration
    # --------------------------------------------------------

    vega_config = construct_network_create_vega_viz(
        organ_nodes,
        organ_edges,
        scenegraph=False,
        show_labels=False,
        filename=f"{organ}_tree",
        root_id=body_root_id
    )

    # --------------------------------------------------------
    # Editorial highlighting: only the specifically requested
    # labeled nodes are recolored. Standard path labels retain
    # their source-derived AS/FTU/CT colors.
    # --------------------------------------------------------

    for row in vega_config["data"][0]["values"]:
        if int(row["id"]) in highlight_label_ids:
            row["color"] = "#C62828"

    # Refuse to save if any biological/tree content changed. This digest
    # covers every node identity, parent edge, name, type, ontology ID,
    # color, and organ value; only layout fields are excluded.
    topology_records = sorted(
        (
            int(row["id"]),
            None if "parent" not in row else int(row["parent"]),
            str(row.get("name", "")),
            str(row.get("type", "")),
            str(row.get("ontology_id", "")),
            str(row.get("color", "")),
            str(row.get("organ", "")),
        )
        for row in vega_config["data"][0]["values"]
    )
    topology_digest = hashlib.sha256(
        json.dumps(
            topology_records,
            separators=(",", ":")
        ).encode("utf8")
    ).hexdigest()
    print(
        "Highlighted topology SHA256:",
        organ,
        topology_digest,
    )

    # ========================================================
    # IMPORTANT:
    # Completely take control of the labels AFTER the
    # constructor returns.
    # ========================================================

    # Map graph ID -> desired label text

    label_text_by_id = {}

    # Body

    label_text_by_id[
        body_root_id
    ] = "Body"

    # Organ root

    organ_root_rows = organ_nodes[
        organ_nodes["graph_int_id"]
        == organ_root_id
    ]

    if len(organ_root_rows) > 0:

        organ_root_name = str(
            organ_root_rows.iloc[0]["name"]
        ).strip()

        label_text_by_id[
            organ_root_id
        ] = organ_root_name.capitalize()

    # FTU representatives

    for ftu_id, node_id in (
        representative_ftu_nodes.items()
    ):

        matching_rows = organ_nodes[
            organ_nodes["graph_int_id"]
            == int(node_id)
        ]

        if len(matching_rows) > 0:

            source_label = str(
                matching_rows.iloc[0]["name"]
            ).strip()

            label_text = FTU_LABEL_OVERRIDES.get(
                ftu_id,
                source_label
            )

            label_text_by_id[
                int(node_id)
            ] = label_text.capitalize()

    # Every intermediate anatomical node on each displayed path. Body,
    # organ, and FTU labels above retain their established wording.
    for _, path_row in organ_nodes[
        organ_nodes["graph_int_id"].isin(label_ids)
    ].iterrows():
        path_node_id = int(path_row["graph_int_id"])
        path_node_name = str(path_row["name"]).strip()
        if not path_node_name or path_node_name.lower() == "nan":
            path_node_name = str(path_row["ontology_id"]).strip()
        label_text_by_id.setdefault(
            path_node_id,
            path_node_name.capitalize()
        )


    # Requested labels override the default source capitalization.
    label_text_by_id.update(highlight_label_text_by_id)

    # --------------------------------------------------------
    # Put an explicit label_text field into Vega data
    # --------------------------------------------------------

    # Angled labels can remain directly anchored to their own nodes.
    label_dx_by_id = {node_id: 7.0 for node_id in label_ids}

    for row in vega_config["data"][0]["values"]:

        row_id = int(row["id"])

        row["label_text"] = (
            label_text_by_id.get(
                row_id,
                ""
            )
        )

        # Keep show_label consistent

        row["show_label"] = (
            1
            if row_id in label_ids
            else 0
        )

        # All displayed labels share the same baseline below their nodes.
        row["label_dx"] = label_dx_by_id.get(row_id, 8.0)
        row["label_dy"] = -7
        row["label_align"] = "left"
        row["label_baseline"] = "bottom"

    # ========================================================
    # FINAL VEGA TEXT CONTROL
    #
    # This prevents the old constructor from making every
    # FTU occurrence visible.
    # ========================================================

    text_mark = vega_config["marks"][-1]

    # Use our explicit label_text instead of the node name

    text_mark["encode"]["enter"]["text"] = {
        "field": "label_text"
    }

    # Show text only when label_text is non-empty

    text_mark["encode"]["update"]["opacity"] = {
        "signal": "datum.label_text != '' ? 1 : 0"
    }

    # Small gap between node and text

    text_mark["encode"]["update"]["dx"] = {
        "field": "label_dx"
    }

    text_mark["encode"]["update"]["dy"] = {
        "field": "label_dy"
    }

    text_mark["encode"]["update"]["align"] = {
        "field": "label_align"
    }

    text_mark["encode"]["update"]["baseline"] = {
        "field": "label_baseline"
    }

    text_mark["encode"]["update"]["angle"] = {
        "value": -30
    }

    displayed_path_label_ids = {
        int(row["id"])
        for row in vega_config["data"][0]["values"]
        if row.get("label_text", "").strip()
    }
    assert displayed_path_label_ids == label_ids, (
        f"{organ}: displayed labels do not exactly match the requested paths"
    )

    displayed_ftu_labels = [
        row["label_text"].strip().lower()
        for row in vega_config["data"][0]["values"]
        if int(row["id"]) in {
            int(node_id)
            for node_id in representative_ftu_nodes.values()
        }
    ]
    assert len(displayed_ftu_labels) == len(set(displayed_ftu_labels)), (
        f"{organ}: duplicate displayed FTU label detected"
    )
    expected_single_ftu_labels = {
        "small_intestine": "intestinal villus",
        "large_intestine": "crypt of lieberkuhn of large intestine",
        "lung": "alveolus of lung",
    }
    if organ in expected_single_ftu_labels:
        expected_label = expected_single_ftu_labels[organ]
        assert displayed_ftu_labels.count(expected_label) == 1, (
            f"{organ}: expected exactly one displayed {expected_label!r} label"
        )


    # --------------------------------------------------------
    # Add the four requested brain labels that are absent from
    # the HRA v2.5 source as a separate editorial annotation
    # block. This preserves the source-derived tree topology.
    # --------------------------------------------------------

    if organ == "brain":
        annotation_top = int(vega_config.get("height", 600)) + 45
        annotation_rows = [
            {
                "label": label,
                "y": annotation_top + (index * 24),
            }
            for index, label in enumerate(BRAIN_EDITORIAL_ANNOTATIONS)
        ]
        vega_config.setdefault("data", []).append(
            {
                "name": "brain_editorial_annotations",
                "values": annotation_rows,
            }
        )
        vega_config.setdefault("marks", []).append(
            {
                "type": "text",
                "from": {"data": "brain_editorial_annotations"},
                "encode": {
                    "enter": {
                        "text": {"field": "label"},
                        "x": {"value": 50},
                        "y": {"field": "y"},
                        "font": {"value": "Benton Sans"},
                        "fontSize": {"value": 12},
                        "fontWeight": {"value": "medium"},
                        "fill": {"value": "#333333"},
                        "align": {"value": "left"},
                        "baseline": {"value": "top"},
                    }
                },
            }
        )
        vega_config["height"] = annotation_top + 120

    # --------------------------------------------------------
    # Save the FINAL patched Vega configuration so the JSON
    # matches the SVG rendered below.
    # --------------------------------------------------------

    final_config_path = (
        Path(output_path)
        / f"{organ}_tree_full_vega_viz_config.json"
    )

    with open(
        final_config_path,
        "w",
        encoding="utf8"
    ) as outfile:
        json.dump(
            vega_config,
            outfile,
            indent=4
        )

    # --------------------------------------------------------
    # Print final labels for debugging
    # --------------------------------------------------------

    print("\nFINAL LABELS:")

    for row in vega_config["data"][0]["values"]:

        if row.get("label_text", "") != "":

            print(
                row["id"],
                "->",
                row["label_text"],
                "|",
                row["ontology_id"]
            )

    # --------------------------------------------------------
    # Render
    # --------------------------------------------------------

    create_vega_viz(
        vega_config,
        f"{organ}_tree",
        scenegraph=False
    )

    print(
        "DONE:",
        organ
    )